# 线段树公共模板：60、4099、2916 与 4086

[原文](https://leetcode.cn/discuss/post/IKZnnt/)

## 公共模板

In [ ]:
# 下标从 0 开始，区间为闭区间；sum 维护和，max 维护最大值。
# tree 已包含当前节点的增量；lazy 记录尚未下传的增量，kth 要求非负整数计数。

In [ ]:
class SegmentTree:
    """区间加、区间查询、单点赋值；sum模式额外支持第k个计数。

    nums非空，下标从0开始，区间均为闭区间，调用者保证下标合法。
    mode='sum'维护区间和，mode='max'维护区间最大值。
    kth要求每个叶子的值都是非负整数，排名k从1开始。
    """

    def __init__(self, nums, mode="sum"):
        if not nums or mode not in ("sum", "max"):
            raise ValueError("nums必须非空，mode必须是sum或max")

        self.n = len(nums)
        self.mode = mode
        self.identity = 0 if mode == "sum" else float("-inf")
        self.merge = (lambda a, b: a + b) if mode == "sum" else max

        self.tree = [self.identity] * (4 * self.n)
        self.lazy = [0] * (4 * self.n)
        self._build(1, 0, self.n - 1, nums)

    def _pull(self, p):
        self.tree[p] = self.merge(
            self.tree[p * 2],
            self.tree[p * 2 + 1]
        )

    def _build(self, p, l, r, nums):
        if l == r:
            self.tree[p] = nums[l]
            return

        mid = (l + r) // 2
        self._build(p * 2, l, mid, nums)
        self._build(p * 2 + 1, mid + 1, r, nums)
        self._pull(p)

    def _apply(self, p, l, r, value):
        # 整段加value：
        # 区间和增加长度*value，区间最大值增加value。
        if self.mode == "sum":
            self.tree[p] += (r - l + 1) * value
        else:
            self.tree[p] += value

        self.lazy[p] += value

    def _push(self, p, l, r):
        if l == r or self.lazy[p] == 0:
            return

        mid = (l + r) // 2
        self._apply(p * 2, l, mid, self.lazy[p])
        self._apply(p * 2 + 1, mid + 1, r, self.lazy[p])
        self.lazy[p] = 0

    def add(self, left, right, value):
        """闭区间[left, right]内每个元素增加value。"""
        if left > right:
            return

        self._add(1, 0, self.n - 1, left, right, value)

    def _add(self, p, l, r, ql, qr, value):
        if ql <= l and r <= qr:
            self._apply(p, l, r, value)
            return

        self._push(p, l, r)
        mid = (l + r) // 2

        if ql <= mid:
            self._add(p * 2, l, mid, ql, qr, value)
        if qr > mid:
            self._add(p * 2 + 1, mid + 1, r, ql, qr, value)

        self._pull(p)

    def query(self, left, right):
        """查询区间和或最大值；空区间返回对应的单位元。"""
        if left > right:
            return self.identity

        return self._query(1, 0, self.n - 1, left, right)

    def _query(self, p, l, r, ql, qr):
        if ql <= l and r <= qr:
            return self.tree[p]

        self._push(p, l, r)
        mid = (l + r) // 2
        result = self.identity

        if ql <= mid:
            result = self.merge(
                result,
                self._query(p * 2, l, mid, ql, qr)
            )
        if qr > mid:
            result = self.merge(
                result,
                self._query(p * 2 + 1, mid + 1, r, ql, qr)
            )

        return result

    def update(self, index, value):
        """单点赋值：将nums[index]改为value。"""
        self._update(1, 0, self.n - 1, index, value)

    def _update(self, p, l, r, index, value):
        if l == r:
            self.tree[p] = value
            self.lazy[p] = 0
            return

        self._push(p, l, r)
        mid = (l + r) // 2

        if index <= mid:
            self._update(p * 2, l, mid, index, value)
        else:
            self._update(p * 2 + 1, mid + 1, r, index, value)

        self._pull(p)

    def kth(self, k):
        """sum模式：返回第k个计数所在的下标；叶子必须为非负整数。"""
        if self.mode != "sum" or not 1 <= k <= self.tree[1]:
            raise ValueError("kth要求sum模式，且1 <= k <= 总计数")

        return self._kth(1, 0, self.n - 1, k)

    def _kth(self, p, l, r, k):
        if l == r:
            return l

        # 区间加可能还未传给孩子，比较左子树计数前必须下推。
        self._push(p, l, r)

        mid = (l + r) // 2
        left_count = self.tree[p * 2]

        if k <= left_count:
            return self._kth(p * 2, l, mid, k)

        return self._kth(
            p * 2 + 1, mid + 1, r, k - left_count
        )

## 60. 排列序列

In [ ]:
# 叶子记录数字是否可用；按阶乘分组计算 rank，再用 kth 选数、update 标记用掉。

In [ ]:
class Solution:
    def getPermutation(self, n: int, k: int) -> str:
        # fact[i]表示i!，包括fact[0]=1
        fact = [1] * (n + 1)
        for i in range(1, n + 1):
            fact[i] = fact[i-1] * i

        # 下标i对应数字i+1；所有数字一开始都可用
        seg = SegmentTree([1] * n)
        answer = []

        # 排列编号转成从0开始，方便整除分组
        k -= 1

        for m in range(n, 0, -1):
            # 剩m个数字时，每个候选数字开头的排列有(m-1)!个
            size = fact[m-1]
            rank = k // size + 1
            k %= size

            # 找出第rank个可用数字，返回的是原下标
            index = seg.kth(rank)
            answer.append(str(index + 1))

            # 选过的数字改成0，后面不会再被选中
            seg.update(index, 0)

        return ''.join(answer)

## 4099. 将锁定数组排序的最少操作次数

In [ ]:
# 先统计必须解锁的位置，超过 k 则无解；其余操作数等于逆序对数。
# 叶子按数值记录已扫描前缀中的出现次数，查询后再把当前数字加入。

In [ ]:
class Solution:
    def minOperations(self, nums: list[int], locked: list[int], k: int) -> int:
        n = len(nums)
        max_num = 0
        lock_cnt = 0

        for i, num in enumerate(nums):
            max_num = max(max_num, num)
            if locked[i]:
                # 自己没放对，或者左边有大数需要跨过自己
                if num != i+1 or max_num > i+1:
                    lock_cnt += 1
                    if lock_cnt > k:
                        return -1

        # 下标num-1记录数字num在已扫描前缀中的出现次数
        seg = SegmentTree([0] * n)
        swap_cnt = 0

        for i, num in enumerate(nums):
            # 前面共有i个数字，减掉其中小于等于num的数量
            swap_cnt += i - seg.query(0, num-1)

            # 当前数字加入已扫描前缀：对应位置加1
            seg.add(num-1, num-1, 1)

        return swap_cnt + lock_cnt

## 2916. 子数组不同元素数目的平方和 II

In [ ]:
# 叶子按左端点记录当前子数组的不同元素数；受影响区间为 [prev+1, r]。
# 先查询旧和，以 2*旧和+长度 更新 ending，再区间加 1 并累加答案。

In [ ]:
class Solution:
    def sumCounts(self, nums: list[int]) -> int:
        mod = 10**9 + 7
        seg = SegmentTree([0] * len(nums))
        last = {}
        ending = 0
        answer = 0

        for r, x in enumerate(nums):
            prev = last.get(x, -1)
            left = prev + 1

            # 先查旧计数之和，计算平方和增加多少
            old_sum = seg.query(left, r)
            ending += 2 * old_sum + (r-left+1)

            # 这些左端点对应的不同元素数，都增加1
            seg.add(left, r, 1)

            # 每个子数组只在它的右端点这一轮被加入总答案
            answer = (answer + ending) % mod
            last[x] = r

        return answer

## 4086. 覆盖目标区间的最少区间

In [ ]:
# 候选按左端点排序，叶子保存右端点；前缀最大值给出当前能接上的最远位置。
# 合并目标并保留全局 covered_end；覆盖实数点，不能跳到 x+1，单点区间也合法。

In [ ]:
from bisect import bisect_right


class Solution:
    def minSelection(self, availableIntervals: list[list[int]],
                     targetIntervals: list[list[int]]) -> int:
        # 合并目标；新建内层列表，不修改传入的目标区间
        sorted_target = sorted(targetIntervals, key=lambda x: x[0])
        merged_target = []

        for start, end in sorted_target:
            if merged_target and start <= merged_target[-1][1]:
                merged_target[-1][1] = max(merged_target[-1][1], end)
            else:
                merged_target.append([start, end])

        # 按左端点排序，叶子存对应候选区间的右端点
        sorted_available = sorted(availableIntervals, key=lambda x: x[0])
        starts = [start for start, end in sorted_available]
        seg = SegmentTree(
            [end for start, end in sorted_available],
            mode="max"
        )

        ans = 0
        covered_end = float('-inf')

        for start, end in merged_target:
            # 目标之间可以跳空；之前覆盖过来的部分继续使用
            cur_start = max(start, covered_end)

            while covered_end < end:
                # 找到所有左端点不超过cur_start的候选
                index = bisect_right(starts, cur_start) - 1

                # 查询这些候选中，最远能到达的右端点
                # 没有候选时index=-1，空区间查询返回负无穷
                next_end = seg.query(0, index)

                if next_end < cur_start or next_end == covered_end:
                    return -1

                ans += 1
                covered_end = next_end
                cur_start = covered_end

        return ans